In [7]:
import pandas as pd
import re

# Paths
in_path  = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\ART.csv"
out_path = r"D:\LinhDao\Programming\SUPERFUNdProject\ArtSuper_Cleaned.csv"
country_lookup_path = r"D:\LinhDao\Programming\SUPERFUNdProject\InternationalCountryCodes.csv"

# Read
df = pd.read_csv(in_path, encoding="cp1252")

# 1) Keep everything up to (but NOT including) the first row that has any cell == "AssetTotal"
if not df.empty:
    mask_asset_total = df.apply(lambda r: r.astype(str).str.strip().eq("AssetTotal").any(), axis=1)
    if mask_asset_total.any():
        cut_idx = mask_asset_total.idxmax()  # first True index
        df = df.loc[:cut_idx-1].copy()

# 2) Ensure columns exist
for col in ["Int/Ext", "Fund Name", "Listed Country", "Value", "Weighting"]:
    if col not in df.columns:
        df[col] = ""

# 3) Row-wise copy: TotalValue -> Value, TotalWeighting -> Weighting (only where present)
if "TotalValue" in df.columns:
    mask_val = df["TotalValue"].notna() & (df["TotalValue"].astype(str).str.strip() != "")
    df.loc[mask_val, "Value"] = df.loc[mask_val, "TotalValue"]

if "TotalWeighting" in df.columns:
    mask_wgt = df["TotalWeighting"].notna() & (df["TotalWeighting"].astype(str).str.strip() != "")
    df.loc[mask_wgt, "Weighting"] = df.loc[mask_wgt, "TotalWeighting"]

# 4) Int/Ext from Type column, then trim those phrases from Type
if "Type" in df.columns:
    type_str = df["Type"].astype(str).str.strip().str.lower()
    df.loc[type_str.str.contains("internally managed", na=False), "Int/Ext"] = 0
    df.loc[type_str.str.contains("externally managed", na=False), "Int/Ext"] = 1

    # Clean Type by removing those tags
    df["Type"] = (
        df["Type"]
        .astype(str)
        .str.replace(r"\b[Ii]nternally\s+[Mm]anaged\b", "", regex=True)
        .str.replace(r"\b[Ee]xternally\s+[Mm]anaged\b", "", regex=True)
        .str.strip()
    )

# 5) Normalize Weighting -> fraction (handles %, commas, numbers)
def _to_fraction(x):
    if pd.isna(x) or str(x).strip() == "":
        return pd.NA
    s = str(x).strip().replace(",", "")
    had_pct = s.endswith("%")
    if had_pct:
        s = s[:-1].strip()
    try:
        v = float(s)
    except:
        return pd.NA
    return v / 100.0 if (had_pct or v > 1) else v

df["Weighting"] = df["Weighting"].apply(_to_fraction)

# 6) Fund Name
df["Fund Name"] = "ArtSuper"

# 7) Listed Country from SecurityIdentifier (skip NaN/blank/"nan"; remove last 2 when taken)
if "SecurityIdentifier" in df.columns:
    def split_sid(val):
        if pd.isna(val):
            return pd.NA, pd.NA
        s = str(val).strip()
        if s == "" or s.lower() == "nan" or len(s) <= 2:
            return s, pd.NA
        return s[:-2], s[-2:]

    sid_split = df["SecurityIdentifier"].apply(split_sid)
    df["SecurityIdentifier"], df["Listed Country"] = zip(*sid_split)

# 8) Drop unwanted columns
to_drop = ["ActualExposure", "EffectOfExposure", "TotalValue", "TotalWeighting", "TotalActualExposure"]
df = df.drop(columns=[c for c in to_drop if c in df.columns], errors="ignore")

# 9) Rename columns BEFORE ordering
rename_map = {
    "AsAtDate": "Effective Date",
    "OptionName": "Option Name",
    "Type": "Asset Class Name",
    "SecurityIdentifier": "Stock ID",
    "UnitsHeld": "Units Held",
    "Ownership": "% Ownership",
    "Value": "Value (AUD)",
    "Name": "Name/Kind of Investment Item",
}
df = df.rename(columns=rename_map)

# === Lookup for Listed Country codes -> full country names ===
try:
    lu = pd.read_csv(country_lookup_path, encoding="cp1252", usecols=[0, 1])
    lu.columns = ["Country", "Code"]
    code2country = dict(zip(lu["Code"].astype(str).str.strip().str.upper(),
                            lu["Country"].astype(str).str.strip()))
    if "Listed Country" in df.columns:
        def map_code_to_country(val):
            if pd.isna(val): return ""
            s = str(val).strip().upper()
            if s == "": return ""
            return code2country.get(s, val)  # keep original if not found
        df["Listed Country"] = df["Listed Country"].apply(map_code_to_country)
except Exception as e:
    print(f"⚠️ Country lookup skipped due to error: {e}")

# 10) Final column order (same as previous files)
final_order = [
    "Effective Date", "Fund Name", "Option Name", "Asset Class Name", "Int/Ext",
    "Name/Kind of Investment Item", "Currency", "Stock ID", "Listed Country",
    "Units Held", "% Ownership", "Address", "Value (AUD)", "Weighting"
]

# Ensure all columns exist; create blanks if missing
for col in final_order:
    if col not in df.columns:
        df[col] = ""

# 11) Fill Effective Date if missing (hardcoded)
df["Effective Date"] = df["Effective Date"].replace("", pd.NA).fillna("31/12/2024")

# 12) Fill empty Int/Ext with 1 and cast to integer
df["Int/Ext"] = df["Int/Ext"].replace("", pd.NA).fillna(1).astype(int)

# 13) Replace empty or "nan" Name/Kind of Investment Item with "Sub Total"
col_name = "Name/Kind of Investment Item"
df[col_name] = df[col_name].apply(
    lambda x: "Sub Total" if pd.isna(x) or str(x).strip().lower() in ["", "nan"] else x
)

# Reorder
df = df[final_order]

# Save
df.to_csv(out_path, index=False, encoding="cp1252")
print(f"✅ Cleaned file saved to: {out_path}")


✅ Cleaned file saved to: D:\LinhDao\Programming\SUPERFUNdProject\ArtSuper_Cleaned.csv


C:\Users\thuon\AppData\Local\Temp\ipykernel_26292\2719032038.py:129: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Int/Ext"] = df["Int/Ext"].replace("", pd.NA).fillna(1).astype(int)


In [ ]:
# new code version aug 20
import pandas as pd
import re

# Paths
in_path  = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\ART.csv"
out_path = r"D:\LinhDao\Programming\SUPERFUNdProject\ArtSuper_Cleaned_final.csv"

# Read
df = pd.read_csv(in_path, encoding="cp1252")

# --- Enforce dtypes before any transformations ---
# Force specific text columns but PRESERVE missing values as NA (avoid "nan" strings)
for col in ["Name", "Currency", "SecurityIdentifier"]:
    if col in df.columns:
        df[col] = df[col].where(df[col].notna(), pd.NA).astype("string")

# Clean Value (remove $ and commas) then convert to numeric
if "Value" in df.columns:
    df["Value"] = (
        df["Value"].astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .replace("", pd.NA)
    )
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

# Convert UnitsHeld to numeric (handle commas/blanks safely)
if "UnitsHeld" in df.columns:
    df["UnitsHeld"] = (
        df["UnitsHeld"].astype(str)
        .str.replace(",", "", regex=False)            # strip thousand-separators
        .replace(["", "nan", "NaN", None], pd.NA)     # normalize empties
    )
    df["UnitsHeld"] = pd.to_numeric(df["UnitsHeld"], errors="coerce")
# --- End dtype enforcement ---

# 1) Keep everything up to (but NOT including) the first row that has any cell == "AssetTotal"
if not df.empty:
    mask_asset_total = df.apply(lambda r: r.astype(str).str.strip().eq("AssetTotal").any(), axis=1)
    if mask_asset_total.any():
        cut_idx = mask_asset_total.idxmax()  # first True index
        df = df.loc[:cut_idx-1].copy()

# 2) Ensure columns exist
for col in ["Int/Ext", "Fund Name", "Listed Country", "Value", "Weighting"]:
    if col not in df.columns:
        df[col] = ""

# 3) Row-wise copy: TotalValue -> Value, TotalWeighting -> Weighting (only where present)
if "TotalValue" in df.columns:
    # Clean $ and commas from TotalValue, convert to numeric BEFORE assigning into Value
    df["TotalValue"] = (
        df["TotalValue"].astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .replace("", pd.NA)
    )
    df["TotalValue"] = pd.to_numeric(df["TotalValue"], errors="coerce")

    mask_val = df["TotalValue"].notna()
    df.loc[mask_val, "Value"] = df.loc[mask_val, "TotalValue"]

if "TotalWeighting" in df.columns:
    mask_wgt = df["TotalWeighting"].notna() & (df["TotalWeighting"].astype(str).str.strip() != "")
    df.loc[mask_wgt, "Weighting"] = df.loc[mask_wgt, "TotalWeighting"]

# 4) Int/Ext from Type column, then trim those phrases from Type
if "Type" in df.columns:
    type_str = df["Type"].astype(str).str.strip().str.lower()
    df.loc[type_str.str.contains("internally managed", na=False), "Int/Ext"] = 0
    df.loc[type_str.str.contains("externally managed", na=False), "Int/Ext"] = 1

    # Clean Type by removing those tags
    df["Type"] = (
        df["Type"]
        .astype(str)
        .str.replace(r"\b[Ii]nternally\s+[Mm]anaged\b", "", regex=True)
        .str.replace(r"\b[Ee]xternally\s+[Mm]anaged\b", "", regex=True)
        .str.strip()
    )

# 5) Normalize Weighting -> fraction (handles %, commas, numbers)
def _to_fraction(x):
    if pd.isna(x) or str(x).strip() == "":
        return pd.NA
    s = str(x).strip().replace(",", "")
    had_pct = s.endswith("%")
    if had_pct:
        s = s[:-1].strip()
    try:
        v = float(s)
    except:
        return pd.NA
    return v / 100.0 if (had_pct or v > 1) else v

df["Weighting"] = df["Weighting"].apply(_to_fraction)

# 6) Fund Name
df["Fund Name"] = "ArtSuper"

# 7) Listed Country from SecurityIdentifier (skip NaN/blank/"nan"; remove last 2 when taken)
if "SecurityIdentifier" in df.columns:
    def split_sid(val):
        if pd.isna(val):
            return pd.NA, pd.NA
        s = str(val).strip()
        if s == "" or s.lower() == "nan" or len(s) <= 2:
            return s, pd.NA
        return s[:-2], s[-2:]

    sid_split = df["SecurityIdentifier"].apply(split_sid)
    df["SecurityIdentifier"], df["Listed Country"] = zip(*sid_split)

# 8) Drop unwanted columns
to_drop = ["ActualExposure", "EffectOfExposure", "TotalValue", "TotalWeighting", "TotalActualExposure"]
df = df.drop(columns=[c for c in to_drop if c in df.columns], errors="ignore")

# 9) Rename columns BEFORE ordering
rename_map = {
    "AsAtDate": "Effective Date",
    "OptionName": "Option Name",
    "Type": "Asset Class Name",
    "SecurityIdentifier": "Stock ID",
    "UnitsHeld": "Units Held",
    "Ownership": "% Ownership",
    "Value": "Value (AUD)",
    "Name": "Name/Kind of Investment Item",
}
df = df.rename(columns=rename_map)

# 10) Final column order (same as previous files)
final_order = [
    "Effective Date", "Fund Name", "Option Name", "Asset Class Name", "Int/Ext",
    "Name/Kind of Investment Item", "Currency", "Stock ID", "Listed Country",
    "Units Held", "% Ownership", "Address", "Value (AUD)", "Weighting"
]

# Ensure all columns exist; create blanks if missing
for col in final_order:
    if col not in df.columns:
        df[col] = ""

# 11) Fill Effective Date if missing (hardcoded)
df["Effective Date"] = df["Effective Date"].replace("", pd.NA).fillna("31/12/2024")

# 12) Fill empty Int/Ext with 1 and cast to integer safely
df["Int/Ext"] = pd.to_numeric(df["Int/Ext"], errors="coerce")
df["Int/Ext"] = df["Int/Ext"].fillna(1).astype(int)

# 13) Replace empty or "nan" Name/Kind of Investment Item with "Sub Total"
col_name = "Name/Kind of Investment Item"
df[col_name] = df[col_name].apply(
    lambda x: "Sub Total" if pd.isna(x) or str(x).strip().lower() in ["", "nan"] else x
)

# Reorder
df = df[final_order]

# --- Hybrid NaN cleanup: blanks for text, keep NaN for numeric (DB-friendly) ---
obj_cols = df.select_dtypes(include=["object", "string"]).columns
df[obj_cols] = df[obj_cols].fillna("")

# Save
df.to_csv(out_path, index=False, encoding="cp1252")
print(f"✅ Cleaned file saved to: {out_path}")


✅ Cleaned file saved to: D:\LinhDao\Programming\SUPERFUNdProject\ArtSuper_Cleaned_final.csv
